In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- Proje Sabitleri ---
DATA_DIR = "../../data/prepared-data" 
TEST_DIR = os.path.join(DATA_DIR, "test")

# Kaydettiğimiz PyTorch (.pth) modellerinin klasörü
MODELS_DIR = "../../models/pytorch"

NUM_CLASSES = 8
IMG_SIZE = (224, 224)
BATCH_SIZE = 64 # Değerlendirme için 64 idealdir

# Cihazı belirle
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# PyTorch modellerini eğitirken kullandığımız standart ImageNet normalizasyonu
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Test seti için transform (ASLA AUGMENTATION OLMAZ)
# Bu, EĞİTİM sırasında 'val' ve 'test' için kullandığımızla aynı olmalı
test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

# Veri setlerini yükle
image_dataset = datasets.ImageFolder(TEST_DIR, test_transform)

# Veri yükleyici (Dataloader) oluştur
# *** shuffle=False OLMASI ZORUNLUDUR ***
test_loader = DataLoader(image_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = image_dataset.classes
print(f"Sınıflar: {class_names}")
print(f"Test verisi: {len(image_dataset)} görüntü")

# --- Gerçek Etiketleri (y_true) Al ---
# ImageFolder, etiketleri .targets özelliğinde doğru sırada tutar
y_true = image_dataset.targets
print(f"Gerçek etiketler (y_true) başarıyla alındı. Toplam: {len(y_true)}")

In [ ]:
# Tüm sonuçları saklamak için bir liste
evaluation_results = []

# Model klasöründeki tüm .pth dosyalarını bul
model_files = [f for f in os.listdir(MODELS_DIR) if f.endswith('.pth')]

print(f"Toplam {len(model_files)} adet eğitilmiş PyTorch modeli bulundu.")

for model_file in model_files:
    print(f"\n{'='*20}")
    print(f"DEĞERLENDİRİLİYOR: {model_file}")
    print(f"{'='*20}")
    
    try:
        # --- 1. Mimariyi yeniden oluştur ---
        # (Dosya ismine göre doğru modeli seçmeliyiz)
        model = None
        
        # 'pretrained=True' veya 'weights=...' KULLANMIYORUZ!
        # Sadece boş mimariyi oluşturuyoruz.
        
        if "resnext50_32x4d" in model_file.lower():
            model = models.resnext50_32x4d(pretrained=False) # Keras'taki 'weights=None' gibi
            num_ftrs = model.fc.in_features
            model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
        
        elif "convnext_tiny" in model_file.lower():
            model = models.convnext_tiny(pretrained=False)
            num_ftrs = model.classifier[-1].in_features
            model.classifier[-1] = nn.Linear(num_ftrs, NUM_CLASSES)
        
        # ... (Gelecekte başka PyTorch modelleri eklersen, buraya 'elif' olarak ekle)
        
        if model is None:
            print(f"Uyarı: {model_file} için uygun mimari bulunamadı (if/elif bloğunu kontrol et). Atlanıyor.")
            continue
            
        # --- 2. Kayıtlı Ağırlıkları Yükle ---
        model_path = os.path.join(MODELS_DIR, model_file)
        # Ağırlıkları (state_dict) boş mimariye yükle
        model.load_state_dict(torch.load(model_path)) 
        model = model.to(DEVICE) # Modeli GPU'ya gönder
        model.eval() # Modeli DEĞERLENDİRME moduna al (Dropout vb. kapat)
        print("Model mimarisi oluşturuldu ve ağırlıklar yüklendi.")

        # --- 3. Tahminleri Topla ---
        all_preds = [] # Tüm tahminleri (y_pred) toplayacağımız liste
        
        print("Test seti üzerinde tahminler yapılıyor...")
        # Gradyan hesaplamasını kapatarak (torch.no_grad) hızlanıyoruz
        with torch.no_grad():
            for inputs, _ in test_loader: # Etiketlere ('_') ihtiyacımız yok, y_true'da var
                inputs = inputs.to(DEVICE)
                
                outputs = model(inputs) # Tahminleri yap
                _, preds = torch.max(outputs, 1) # En yüksek skorlu sınıfı al
                
                all_preds.extend(preds.cpu().numpy()) # Tahminleri CPU'ya geri alıp listeye ekle

        # --- 4. Metrikleri Hesapla ---
        accuracy = accuracy_score(y_true, all_preds)
        report_dict = classification_report(y_true, all_preds, target_names=class_names, output_dict=True)
        
        print(f"\n--- {model_file} Sonuçları ---")
        print(f"Genel Doğruluk (Accuracy): {accuracy * 100:.2f}%")
        print("\nSınıflandırma Raporu (Classification Report):")
        print(classification_report(y_true, all_preds, target_names=class_names))
        
        # Sonuçları ana tablo için kaydet
        evaluation_results.append({
            "Model": model_file.replace('.pth', ''),
            "Accuracy": accuracy,
            "F1-Score (Weighted)": report_dict['weighted avg']['f1-score'],
            "Precision (Weighted)": report_dict['weighted avg']['precision'],
            "Recall (Weighted)": report_dict['weighted avg']['recall']
        })

        # --- 5. Karmaşıklık Matrisini (Confusion Matrix) Çizdir ---
        cm = confusion_matrix(y_true, all_preds)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Karmaşıklık Matrisi - {model_file}')
        plt.xlabel('Tahmin Edilen (Predicted)')
        plt.ylabel('Gerçek (True)')
        plt.show()

    except Exception as e:
        print(f"!!! MODEL {model_file} YÜKLENİRKEN/DEĞERLENDİRİLİRKEN HATA OLUŞTU: {e}")

print("\n\n--- DEĞERLENDİRME TAMAMLANDI ---")

In [ ]:
# Sonuçları Pandas DataFrame'e çevir
results_df = pd.DataFrame(evaluation_results)
results_df = results_df.set_index("Model") # Model ismini index yap

# F1-Skoruna göre sırala
results_df = results_df.sort_values(by="F1-Score (Weighted)", ascending=False)

print("--- NİHAİ MODEL KARŞILAŞTIRMA TABLOSU (PyTorch - Test Seti Sonuçları) ---")
print(results_df.to_markdown(floatfmt=".4f"))